In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import os
import time
import numpy as np
import pandas as pd
import random
import re
import string
import matplotlib.pyplot as plt

2025-12-04 19:57:12.287194: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-04 19:57:12.328075: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764849432.352162 1813654 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764849432.359327 1813654 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764849432.377902 1813654 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Restrict TensorFlow to only use the first GPU
        tf.config.experimental.set_visible_devices(gpus[3], 'GPU')

        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

4 Physical GPUs, 1 Logical GPUs


I0000 00:00:1764849441.418117 1813654 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14207 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:b3:00.0, compute capability: 8.9


In [3]:
def load_mapping(path, key_to_int=False, val_to_int=False):
    arr = np.load(path, allow_pickle=True)

    if isinstance(arr, np.ndarray) and arr.ndim == 0:
        arr = arr.item()

    if isinstance(arr, np.ndarray):
        try:
            arr = dict(arr)         
        except Exception:
            arr = dict(arr.tolist())

    if not isinstance(arr, dict):
        raise ValueError(f"Unsupported mapping format: {type(arr)}")

    out = {}
    for k, v in arr.items():
        k_str = str(k).replace("'", "").strip()
        v_str = str(v).replace("'", "").strip()

        if key_to_int:
            try:
                k_str = int(k_str)
            except:
                continue
        if val_to_int:
            try:
                v_str = int(v_str)
            except:
                continue

        out[k_str] = v_str
    return out

dictionary_path = "./dictionary"
word2Id_dict = load_mapping(dictionary_path + "/word2Id.npy", key_to_int=False, val_to_int=True)
id2word_dict = load_mapping(dictionary_path + "/id2Word.npy", key_to_int=True, val_to_int=False)

pad_id = int(word2Id_dict['<PAD>'])
rare_id = int(word2Id_dict['<RARE>'])

id2word_dict[pad_id] = '<PAD>'
id2word_dict[rare_id] = '<RARE>'

print("word2Id example:", list(word2Id_dict.items())[:3])
print("id2Word example:", list(id2word_dict.items())[:3])

word2Id example: [('<PAD>', 5427), ('flower', 1), ('petals', 2)]
id2Word example: [(0, '<PAD>'), (1, 'flower'), (2, 'petals')]


In [4]:
hparas = {
    'MAX_SEQ_LENGTH': 20,
    'EMBED_DIM': 128,
    'VOCAB_SIZE': len(word2Id_dict),
    'RNN_HIDDEN_SIZE': 256,
    'Z_DIM': 100,
    'IMAGE_SIZE': [64, 64, 3],
    'BATCH_SIZE': 256,

    'LR_G': 1e-4,
    'LR_D': 2e-4,
    'BETA_1': 0.0,
    'BETA_2': 0.9,

    'LAMBDA_GP': 10.0,
    'N_EPOCH': 200,      
    'N_CRITIC': 3,      
    'CHECKPOINTS_DIR': './checkpoints_film', 
}

In [6]:
def safe_decode(ids):
    words = []
    for w in ids:
        w = int(w)
        word = id2word_dict.get(w, "<UNK>")
        if word == "<PAD>":
            continue
        words.append(word)
    return " ".join(words)

In [7]:
import tensorflow_hub as hub

class USETextEncoder(tf.keras.Model):
    def __init__(self, hparas):
        super().__init__()
        self.encoder = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=False,
            dtype=tf.string
        )

    def call(self, sentences):
        return self.encoder(sentences)  # (batch, 512)
    
text_encoder = USETextEncoder(hparas)

/home/kevin110062222/miniconda3/envs/py310/lib/python3.10/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [8]:
def precompute_embeddings(pkl_file, save_path, batch=256):
    df = pd.read_pickle(pkl_file)
    captions = df["Captions"].values
    image_paths = df["ImagePath"].values

    all_caps, all_paths = [], []
    for caps, path in zip(captions, image_paths):
        for cap in caps:
            if isinstance(cap, (list, np.ndarray, pd.Series)):
                sent = safe_decode(cap[:hparas['MAX_SEQ_LENGTH']])
            elif isinstance(cap, str):
                sent = cap
            else:
                sent = str(cap)

            if sent.strip() == "":
                sent = "<PAD>"

            all_caps.append(sent)
            all_paths.append(path)

    embeds = []
    for i in range(0, len(all_caps), batch):
        cap_batch = tf.constant(all_caps[i:i+batch], dtype=tf.string)
        emb_batch = text_encoder(cap_batch)  # (b, 512)
        embeds.append(emb_batch.numpy().astype(np.float32))

    all_embeds = np.concatenate(embeds, axis=0)
    all_paths = np.array(all_paths, dtype=object)

    np.savez(save_path, embeds=all_embeds, paths=all_paths)
    print("saved:", save_path, all_embeds.shape, all_paths.shape)

precompute_embeddings(
    "./dataset/text2ImgData.pkl",
    "./dataset/train_use_embed.npz",
    batch=256
)

saved: ./dataset/train_use_embed.npz (70504, 512) (70504,)


In [9]:
def dataset_generator_from_embed(npz_path, batch_size):
    pack = np.load(npz_path, allow_pickle=True)
    all_embeds = pack["embeds"]   # float32 (N, 512)
    all_paths  = pack["paths"]    # object  (N,)

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_embeds))

    def map_fn(image_path, text_embed):
        img = tf.io.read_file(image_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img.set_shape([None, None, 3])
        img = tf.image.convert_image_dtype(img, tf.float32)

        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.05)

        img = tf.image.resize(img, [72, 72])
        img = tf.image.random_crop(img, [64, 64, 3])
        img = (img - 0.5) * 2.0

        text_embed = tf.math.l2_normalize(text_embed, axis=-1)

        return img, text_embed


    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.repeat()
    ds = ds.shuffle(5000)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [10]:
data_path = './dataset'

train_dataset = dataset_generator_from_embed(
    data_path + "/train_use_embed.npz",
    hparas["BATCH_SIZE"]
)

In [11]:
class Generator(Model):
    def __init__(self, hparas):
        super().__init__()

        self.text_proj = layers.Dense(256, activation=None)

        self.fc = layers.Dense(
            4 * 4 * 512,
            use_bias=False,
            kernel_initializer='orthogonal'
        )
        self.ln0 = layers.LayerNormalization()
        self.act0 = layers.LeakyReLU(alpha=0.2)

        def up_block(filters):
            return tf.keras.Sequential([
                layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
                layers.Conv2D(
                    filters, 3, padding='same', use_bias=False,
                    kernel_initializer='orthogonal'
                ),
                layers.LayerNormalization(),
                layers.LeakyReLU(alpha=0.2)
            ])

        self.up1 = up_block(256)
        self.up2 = up_block(128)
        self.up3 = up_block(64)
        self.up4 = up_block(32)

        self.to_rgb = layers.Conv2D(
            3, 3, padding='same', activation='tanh',
            kernel_initializer='orthogonal'
        )

        self.film1_gamma = layers.Dense(256)
        self.film1_beta  = layers.Dense(256)
        self.film2_gamma = layers.Dense(128)
        self.film2_beta  = layers.Dense(128)
        self.film3_gamma = layers.Dense(64)
        self.film3_beta  = layers.Dense(64)
        self.film4_gamma = layers.Dense(32)
        self.film4_beta  = layers.Dense(32)

    def apply_film(self, x, t, gamma_layer, beta_layer):
        gamma = gamma_layer(t)
        beta = beta_layer(t)
        c = x.shape[-1]
        gamma = tf.reshape(gamma, [-1, 1, 1, c])
        beta  = tf.reshape(beta,  [-1, 1, 1, c])
        return x * (1.0 + gamma) + beta

    def call(self, text_embed, noise, training=False):
        t = self.text_proj(text_embed)
        t = tf.nn.leaky_relu(t, alpha=0.2)
        t = tf.math.l2_normalize(t, axis=-1)

        x = tf.concat([noise, t], axis=1)

        x = self.fc(x)
        x = self.ln0(x)
        x = self.act0(x)
        x = tf.reshape(x, (-1, 4, 4, 512))

        x = self.up1(x, training=training)
        x = self.apply_film(x, t, self.film1_gamma, self.film1_beta)

        x = self.up2(x, training=training)
        x = self.apply_film(x, t, self.film2_gamma, self.film2_beta)

        x = self.up3(x, training=training)
        x = self.apply_film(x, t, self.film3_gamma, self.film3_beta)

        x = self.up4(x, training=training)
        x = self.apply_film(x, t, self.film4_gamma, self.film4_beta)

        x = self.to_rgb(x)
        return x


In [12]:
class Discriminator(Model):
    def __init__(self, hparas):
        super(Discriminator, self).__init__()
        
        self.conv1 = layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same')
        self.lrelu1 = layers.LeakyReLU(alpha=0.2)
        
        self.conv2 = layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same')
        # self.ln2 = layers.LayerNormalization() # BN -> LN
        self.lrelu2 = layers.LeakyReLU(alpha=0.2)
        
        self.conv3 = layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same')
        # self.ln3 = layers.LayerNormalization()
        self.lrelu3 = layers.LeakyReLU(alpha=0.2)
        
        self.conv4 = layers.Conv2D(512, (4, 4), strides=(2, 2), padding='same')
        # self.ln4 = layers.LayerNormalization() 
        self.lrelu4 = layers.LeakyReLU(alpha=0.2)
        
        self.text_dense = layers.Dense(256) 
        
        self.conv_last = layers.Conv2D(512, (1, 1), padding='same')
        # self.ln_last = layers.LayerNormalization()
        self.lrelu_last = layers.LeakyReLU(alpha=0.2)
        
        self.flatten = layers.Flatten()
        self.final_dense = layers.Dense(1) 

    def call(self, img, text_embed):
        x = self.conv1(img)
        x = self.lrelu1(x)

        x = self.conv2(x)
        x = self.lrelu2(x)

        x = self.conv3(x)
        x = self.lrelu3(x)

        x = self.conv4(x)
        x = self.lrelu4(x)

        text_embed = tf.math.l2_normalize(text_embed, axis=-1)
        t = self.text_dense(text_embed)
        t = tf.nn.leaky_relu(t, alpha=0.2)

        t = tf.reshape(t, [-1, 1, 1, 256])
        t = tf.tile(t, [1, 4, 4, 1])

        concat = tf.concat([x, t], axis=3)

        out = self.conv_last(concat)
        out = self.lrelu_last(out)

        out = self.flatten(out)
        logits = self.final_dense(out)
        return logits

In [13]:
generator = Generator(hparas)
discriminator = Discriminator(hparas)

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=hparas['LR_G'],
    beta_1=hparas['BETA_1'],
    beta_2=hparas['BETA_2']
)
discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=hparas['LR_D'],
    beta_1=hparas['BETA_1'],
    beta_2=hparas['BETA_2']
)

/home/kevin110062222/miniconda3/envs/py310/lib/python3.10/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [14]:
def gradient_penalty(discriminator, real_images, fake_images, text_embed):
    batch_size = tf.shape(real_images)[0]
    alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
    diff = fake_images - real_images
    interpolated = real_images + alpha * diff
    
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolated)
        pred = discriminator(interpolated, text_embed)
    
    grads = gp_tape.gradient(pred, [interpolated])[0]
    
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-6) 
    
    gp = tf.reduce_mean((norm - 1.0) ** 2)
    return gp

In [15]:
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator,
    text_encoder=text_encoder
)

ckpt_manager = tf.train.CheckpointManager(
    checkpoint,
    hparas['CHECKPOINTS_DIR'],
    max_to_keep=20
)

In [16]:
@tf.function
def train_discriminator_step(real_image, text_embed):
    noise = tf.random.normal([hparas['BATCH_SIZE'], hparas['Z_DIM']])

    with tf.GradientTape() as disc_tape:
        text_embed = tf.stop_gradient(text_embed)  
        fake_image = generator(text_embed, noise, training=False)

        real_logits = discriminator(real_image, text_embed, training=True)
        fake_logits = discriminator(fake_image, text_embed, training=True)

        w_loss = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits)

        gp = gradient_penalty(discriminator, real_image, fake_image, text_embed)
        d_loss = w_loss + hparas['LAMBDA_GP'] * gp

        # tf.print("w_loss", w_loss, "gp", gp, "real", tf.reduce_mean(real_logits), "fake", tf.reduce_mean(fake_logits))


    grad_d = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(zip(grad_d, discriminator.trainable_variables))
    return d_loss


In [17]:
@tf.function
def train_generator_step(text_embed):
    noise = tf.random.normal([hparas['BATCH_SIZE'], hparas['Z_DIM']])

    with tf.GradientTape() as gen_tape:
        text_embed = tf.stop_gradient(text_embed)
        fake_image = generator(text_embed, noise, training=True)
        fake_logits = discriminator(fake_image, text_embed, training=False)
        g_loss = -tf.reduce_mean(fake_logits)

    grad_g = gen_tape.gradient(g_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(grad_g, generator.trainable_variables))
    return g_loss


In [18]:
test_captions = [
    "red flower with yellow center",
    "blue flower with white edges",
    "yellow flower with orange center"
]
fixed_cap_tensor = tf.constant(test_captions, dtype=tf.string)
fixed_text_embed = text_encoder(fixed_cap_tensor)                 # (3,512)
fixed_noise = tf.random.normal([len(test_captions), hparas["Z_DIM"]], seed=1234)

def generate_sample_images(epoch):
    fig, axes = plt.subplots(1, len(test_captions), figsize=(15,5))
    for i, cap_text in enumerate(test_captions):
        te = fixed_text_embed[i:i+1]
        nz = fixed_noise[i:i+1]

        fake_image = generator(te, nz, training=False)
        img_array = fake_image[0].numpy() * 0.5 + 0.5
        img_array = np.clip(img_array, 0, 1)

        axes[i].imshow(img_array)
        axes[i].set_title(cap_text, fontsize=8)
        axes[i].axis("off")

    plt.tight_layout()
    plt.savefig(f"./samples/epoch_{epoch:03d}.png")
    plt.close()

In [19]:
def inference(checkpoint_path, k_candidates=1):
    seed = 5001

    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    checkpoint.restore(checkpoint_path).expect_partial()
    print(f"Checkpoint loaded: {checkpoint_path}")

    test_data = pd.read_pickle(data_path + "/testData.pkl")
    test_captions = test_data["Captions"].values
    test_ids = test_data["ID"].values

    os.makedirs("./inference", exist_ok=True)

    for i, cap_raw in enumerate(test_captions):

        if isinstance(cap_raw, (list, np.ndarray, pd.Series)):
            cap_text = safe_decode(cap_raw[:20])
        elif isinstance(cap_raw, str):
            cap_text = cap_raw
        else:
            cap_text = str(cap_raw)

        if cap_text.strip() == "":
            cap_text = "<PAD>"

        cap_tensor = tf.constant([cap_text], dtype=tf.string)
        text_embed_1 = text_encoder(cap_tensor)          # (1,512)
        text_embed_k = tf.tile(text_embed_1, [k_candidates, 1])  # (K,512)

        noise = tf.random.normal([k_candidates, hparas["Z_DIM"]])

        fake_images = generator(text_embed_k, noise, training=False)  # (K,64,64,3)

        logits = discriminator(fake_images, text_embed_k)
        logits = tf.squeeze(logits, axis=-1)  # (K,)

        best_idx = tf.argmax(logits)
        best_img = fake_images[best_idx]

        img_array = best_img.numpy() * 0.5 + 0.5
        img_array = np.clip(img_array, 0, 1)

        save_path = f"./inference/inference_{int(test_ids[i]):04d}.jpg"
        plt.imsave(save_path, img_array)

        if (i+1) % 50 == 0:
            print(f"Generated {i+1}/{len(test_ids)} images")

    print(f"Inference done. {len(test_ids)} images saved.")


In [20]:
def train(dataset, epochs):
    print("Start Training WGAN-GP...")
    if not os.path.exists(hparas['CHECKPOINTS_DIR']):
        os.makedirs(hparas['CHECKPOINTS_DIR'])
    if not os.path.exists('./samples'):
        os.makedirs('./samples')
        
    df_temp = pd.read_pickle(data_path + '/text2ImgData.pkl')
    num_samples = sum(len(caps) for caps in df_temp["Captions"].values)
    steps_per_epoch = num_samples // hparas['BATCH_SIZE']
    if steps_per_epoch == 0: steps_per_epoch = 1
    print(f"Total Samples: {num_samples}, Batch: {hparas['BATCH_SIZE']}, Steps/Epoch: {steps_per_epoch}")

    data_iter = iter(dataset)

    for epoch in range(epochs):
        start = time.time()
        g_loss_avg = 0
        d_loss_avg = 0
        
        for step in range(steps_per_epoch):
            for _ in range(hparas['N_CRITIC']):
                try:
                    image, text_embed = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataset)
                    image, text_embed = next(data_iter)
                
                d_loss = train_discriminator_step(image, text_embed)

            g_loss = train_generator_step(text_embed)
            
            g_loss_avg += g_loss
            d_loss_avg += d_loss
            
        print(f'Epoch {epoch+1}, Gen Loss: {g_loss_avg/steps_per_epoch:.4f}, Disc Loss: {d_loss_avg/steps_per_epoch:.4f}, Time: {time.time()-start:.2f}s')
        
        if (epoch + 1) % 5 == 0:
            ckpt_manager.save(checkpoint_number=epoch+1) 
        generate_sample_images(epoch + 1)

In [21]:
train(train_dataset, hparas['N_EPOCH'])

print("Training completed, ready for inference...")

Start Training WGAN-GP...
Total Samples: 70504, Batch: 256, Steps/Epoch: 275


I0000 00:00:1764849479.118301 1814130 cuda_dnn.cc:529] Loaded cuDNN version 91600


KeyboardInterrupt: 

In [ ]:
ckpt_path = os.path.join(hparas['CHECKPOINTS_DIR'], "ckpt-140")

if tf.io.gfile.exists(ckpt_path + ".index"):
    print(f"Using weights file: {ckpt_path}")
    inference(ckpt_path, k_candidates=4)    
else:
    print("Error: Weights file not found:", ckpt_path)

Using weights file: ./checkpoints_film/ckpt-140
Checkpoint loaded: ./checkpoints_film/ckpt-140
Generated 50/819 images
Generated 100/819 images
Generated 150/819 images
Generated 200/819 images
Generated 250/819 images
Generated 300/819 images
Generated 350/819 images
Generated 400/819 images
Generated 450/819 images
Generated 500/819 images
Generated 550/819 images
Generated 600/819 images
Generated 650/819 images
Generated 700/819 images
Generated 750/819 images
Generated 800/819 images
Inference done. 819 images saved.


In [ ]:
%cd testing
!python inception_score.py ../inference ../score.csv 39 .
%cd ..

/ssd2/kevin/comp3/testing
2025-11-27 15:19:14.200361: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-27 15:19:14.272545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-27 15:19:16.253165: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1764227957.853446 1505851 gpu_device.cc:20

In [ ]:
import os
score_file = './score.csv'

if os.path.exists(score_file):
    df_score = pd.read_csv(score_file)
    mean_score = np.mean(df_score['score'].values)
    print(f'Mean Score: {mean_score:f}')
else:
    print('Score file not found.')

Mean Score: 0.465432


In [ ]:
import textwrap

def wrap_caption(text, width=40):
    lines = textwrap.wrap(text, width=width)
    return "\n".join(lines)

def generate_5caps_5noises_table(checkpoint_path, num_caps=5, num_noises=5):
    checkpoint.restore(checkpoint_path).expect_partial()
    print(f"[Grid] Loaded checkpoint: {checkpoint_path}")

    test_data_path = os.path.join(data_path, "testData.pkl")
    df_test = pd.read_pickle(test_data_path)
    n_total = len(df_test)

    indices = np.random.choice(n_total, size=min(num_caps, n_total), replace=False)

    data_rows = len(indices)
    cols = num_noises + 1      
    rows = data_rows + 1      

    height_ratios = [0.45] + [1.0] * data_rows

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(cols * 2.2, (data_rows + 0.3) * 2.2),
        gridspec_kw={
            "width_ratios": [3.0] + [1.0] * num_noises,
            "height_ratios": height_ratios,
        },
    )

    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    header_labels = ["Test Caption"] + [f"Noise{i}" for i in range(1, num_noises + 1)]
    for j in range(cols):
        ax_h = axes[0, j]
        ax_h.axis("off")
        ax_h.text(
            0.5, 0.5,
            header_labels[j],
            fontsize=16,
            fontweight="bold",
            ha="center",
            va="center",
            transform=ax_h.transAxes,
        )

    print("Selected test captions:")
    print("-" * 60)

    for row_idx, idx in enumerate(indices):
        row = df_test.iloc[idx]
        img_id = row["ID"]
        caps_raw = row["Captions"]

        if isinstance(caps_raw, (list, np.ndarray, pd.Series)):
            if len(caps_raw) > 0 and isinstance(caps_raw[0], (list, np.ndarray, pd.Series)):
                cap_ids = caps_raw[0]
            else:
                cap_ids = caps_raw
            cap_text = safe_decode(cap_ids[:hparas["MAX_SEQ_LENGTH"]])
        elif isinstance(caps_raw, str):
            cap_text = caps_raw
        else:
            cap_text = str(caps_raw)

        if cap_text.strip() == "":
            cap_text = "<PAD>"

        full_caption = f"[ID {int(img_id):04d}]\n" + wrap_caption(cap_text, width=45)
        print(full_caption)
        print("-" * 60)

        r = row_idx + 1

        ax_text = axes[r, 0]
        ax_text.axis("off")
        ax_text.text(
            0.02, 0.5,
            full_caption,
            fontsize=14,
            ha="left",
            va="center",
            wrap=True,
            transform=ax_text.transAxes,
        )

        cap_tensor = tf.constant([cap_text], dtype=tf.string)
        text_embed_1 = text_encoder(cap_tensor)
        text_embed_k = tf.tile(text_embed_1, [num_noises, 1])

        noise = tf.random.normal([num_noises, hparas["Z_DIM"]])
        fake_images = generator(text_embed_k, noise, training=False).numpy()
        fake_images = np.clip(fake_images * 0.5 + 0.5, 0.0, 1.0)

        for j in range(num_noises):
            ax = axes[r, j + 1]
            ax.imshow(fake_images[j])
            ax.axis("off")

    for r in range(rows):
        for c in range(cols):
            ax = axes[r, c]
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(1.0)
                spine.set_edgecolor("black")

    plt.subplots_adjust(
        left=0.02,
        right=0.98,
        top=0.98,
        bottom=0.02,
        wspace=0.0,
        hspace=0.0,
    )

    os.makedirs("./samples", exist_ok=True)
    out_path = "./samples/5captions_5noises_table.png"
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"[Grid] Saved figure to {out_path}")

In [ ]:
ckpt_path = os.path.join(hparas['CHECKPOINTS_DIR'], "ckpt-195")
if tf.io.gfile.exists(ckpt_path + ".index"):
    generate_5caps_5noises_table(ckpt_path, num_caps=5, num_noises=5)